In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from utils.logger import get_logger

class EvalAgentBuilder:
    def __init__(self, model_name="gpt-4"):
        self.judge_model = init_chat_model(model_name)

    def build_agent(self):
        def eval_node(state):
            get_logger("EvalAgent").info("Eval Node")

            query = state["messages"][0].content
            answer = state["messages"][-1].content
            # context retrieval can be injected or passed in
            context_docs = state.get("context_docs", [])
            context = "\n\n".join(doc.page_content for doc in context_docs)

            prompt = f"""
            You are an evaluator. Given the user query, retrieved context, and the answer:

            Query: {query}
            Context: {context}
            Answer: {answer}

            Evaluate the following metrics on a scale of 1 (poor) to 5 (excellent):

            1. Groundedness: Is the answer supported by the provided context?
            2. Relevance: Is the answer relevant to the query?

            Return your evaluation as JSON with keys 'groundedness' and 'relevance'.
            """

            result = self.judge_model.invoke(prompt)
            eval_message = HumanMessage(content=result.content, name="eval_agent")

            return {
                "messages": state["messages"] + [eval_message],
                "goto": "END"
            }

        return eval_node

In [ ]:
class GraphInstance:
    def __init__(self):
        self.vector_db = DataIngestor().ingest(load_existing=True)
        self.rag_agent = RAGAgentBuilder(self.vector_db).build_agent()
        self.eval_agent = EvalAgentBuilder().build_agent()

    def workflow(self):
        workflow = StateGraph(MessagesState)
        workflow.add_node("rag_agent", self.rag_agent)
        workflow.add_node("eval_agent", self.eval_agent)
        workflow.add_edge(START, "rag_agent")
        workflow.add_edge("rag_agent", "eval_agent")
        return workflow.compile()

In [ ]:
def rag_node(self, state: MessagesState) -> Command[Literal["eval_agent", END]]:
    get_logger("Graph").info("Rag Node")
    result = self.rag_agent.invoke(state)

    # Retrieve docs explicitly
    query = state["messages"][0].content
    context_docs = self.vector_db.similarity_search(query, k=3)

    goto = self.get_next_node(result["messages"][-1], "eval_agent")
    result["messages"][-1] = HumanMessage(
        content=result["messages"][-1].content, name="rag_agent"
    )

    return Command(
        update={
            "messages": result["messages"],
            "context_docs": context_docs   # <-- pass docs forward
        },
        goto=goto
    )